# Notebook 12 — Statistics, robustness, and report

Predeclared mixed-effects diagonal model, bootstrap CI, search-budget-matched permutation test, and final report assembly.

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="12", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220752_smoke_eagle_12_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
Fit the diagonal model on a planted target-specific effect; run a budget-matched permutation test.

In [4]:
from subliminal_icl import statistics as st
t=[];j=[];q=[];d=[]
rng=np.random.default_rng(7)
for tt in S.ANIMALS[:5]:
  for jj in S.ANIMALS[:5]:
    for qi in range(6):
      t.append(tt); j.append(jj); q.append(qi)
      d.append((1.2 if tt==jj else 0.0) + rng.normal(0,0.2))
res = st.fit_diagonal_model(st.DiagonalObservations(t=t,j=j,q=q,delta=d), n_boot=400)
# budget-matched permutation null (max over each null search)
null_max = [rng.normal(0,0.3) for _ in range(50)]
perm = st.permutation_test(res["gamma"], null_max)
print("gamma:", round(res["gamma"],3), "CI excl0:", res["gamma_excludes_zero"], "perm p:", round(perm["p_value"],4))

gamma: 1.228 CI excl0: True perm p: 0.0196


## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("gamma_positive_ci", res["gamma"] > 0 and res["gamma_excludes_zero"], "gamma>0, CI excludes 0"),
          ("permutation_significant", perm["p_value"] < 0.05, "budget-matched permutation p<0.05")]
gs = run_gate_checks("gate_12_statistics", "Diagonal gamma + permutation", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,gamma_positive_ci,PASS,"gamma>0, CI excludes 0"
1,permutation_significant,PASS,budget-matched permutation p<0.05


GATE gate_12_statistics -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.